# Under the Hood: Loading & Inspecting Open-Source LLMs

📓 **Notebook:** [Open in Colab](https://colab.research.google.com/drive/1tD2Fj89RJUjdbOc2c5uIaTtdXItj4_mq#scrollTo=E0Jits1f07vj)

## What this notebook covers

This notebook goes one level below the `pipeline()` API and works directly with the
lower-level HuggingFace `transformers` classes — `AutoTokenizer` and
`AutoModelForCausalLM` — to load, quantize, run, and inspect real open-source LLMs
on a free T4 GPU.

**Models loaded and compared:**
- Llama 3.2 (1B-Instruct) / Llama 3.1 (8B-Instruct)
- Phi-4-mini-instruct
- Gemma 3 (270M-it)
- Qwen 3 (4B-Instruct)
- DeepSeek-R1-Distill-Qwen (1.5B)

## What was actually done

1. **Authentication** — Logged into HuggingFace via `HF_TOKEN` and requested gated
   access to Llama and Gemma model weights.
2. **4-bit Quantization** — Configured `BitsAndBytesConfig` (NF4, double quant,
   `bfloat16` compute dtype) so an 8B-parameter model fits comfortably in T4 memory.
3. **Tokenization** — Used `apply_chat_template()` to convert chat-style messages
   into model-ready input IDs, and compared `add_generation_prompt=True/False` to see
   how it changes whether the model *answers* vs. just *continues* the prompt.
4. **Inference** — Ran `model.generate()` first in blocking mode, then rewrote it
   using `TextStreamer` so tokens stream back live instead of appearing all at once.
5. **Architecture inspection** — Printed the raw `model` object to see the actual
   PyTorch layer stack: embedding layer → repeated Decoder layers (each with
   self-attention + MLP + norm sublayers) → final LM head.
6. **Memory hygiene** — Explicitly `del`'d models/tensors and called
   `gc.collect()` + `torch.cuda.empty_cache()` between model loads to avoid OOM
   errors on the shared T4.
7. **Generalized helper** — Wrapped the whole load→tokenize→generate flow into a
   single `generate(model, messages, quant, max_new_tokens)` function, complete with
   an explicit `attention_mask`, to quickly swap between five different model
   families with one line of code.

## Why this matters / learning value

- **Demystifies the "black box"** — seeing the actual decoder-layer stack printed
  out makes concepts like embeddings, attention, and MLP concrete instead of
  abstract theory.
- **Quantization intuition** — directly observing `get_memory_footprint()` before/after
  4-bit quantization makes the cost/performance trade-off tangible, not just a
  config flag.
- **Chat templates matter** — the `add_generation_prompt` experiment shows *why*
  raw next-token prediction ≠ instruction-following, which is foundational for
  anything downstream (fine-tuning, agents, RAG).
- **Attention masks explained by doing** — writing an explicit `attention_mask`
  (even as all-1s for single prompts) builds the right mental model for when
  real batching/padding shows up later.
- **Model-agnostic muscle memory** — writing one `generate()` function that works
  across Llama/Phi/Gemma/Qwen/DeepSeek builds the habit of treating model choice as
  a swappable parameter, not a hardcoded assumption — directly useful for later
  eval/benchmarking work.
- **Production hygiene early** — the explicit memory cleanup pattern is a habit
  worth carrying into any Colab/Kaggle workflow that loads multiple large models
  in sequence.

## Key takeaways to carry forward

- Every Decoder layer = **Self-Attention** (token-to-token mixing) + **MLP**
  (per-token transformation, usually the largest parameter chunk) + normalization.
- `BitsAndBytesConfig(nf4, double_quant)` is the standard recipe for running
  8B-class models on a free T4.
- `attention_mask` tells the model which tokens are real vs. padding — trivial
  (all 1s) for single prompts, essential once batching multiple prompts together.
- Always clean up (`del` + `gc.collect()` + `empty_cache()`) before loading the
  next model in the same session.